In [1]:
import pandas as pd

# Load the dataset
df = pd.read_csv('titanic.csv')

# Print schema summary
print("Schema Summary:")
for col in df.columns:
    print(f"{col}: {df[col].dtype}, missing: {df[col].isnull().sum()}")

# Clean the dataset
# Drop specified columns
dropped_cols = ['PassengerId', 'Name', 'Ticket', 'Cabin']
df.drop(dropped_cols, axis=1, inplace=True)
print(f"Dropped columns: {', '.join(dropped_cols)}")

# Impute missing Age with median
missing_age = df['Age'].isnull().sum()
median_age = df['Age'].median()
df['Age'].fillna(median_age, inplace=True)
print(f"Imputed {missing_age} missing Age values with median: {median_age}")

# Impute missing Embarked with mode
missing_embarked = df['Embarked'].isnull().sum()
mode_embarked = df['Embarked'].mode()[0]
df['Embarked'].fillna(mode_embarked, inplace=True)
print(f"Imputed {missing_embarked} missing Embarked values with mode: '{mode_embarked}'")

# Encode Sex as binary
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
print("Encoded Sex column as binary: male=0, female=1")

# Print final shape
print(f"Final dataframe shape: {df.shape}")

# Save cleaned dataframe
df.to_csv('titanic_clean.csv', index=False)


Schema Summary:
PassengerId: int64, missing: 0
Survived: int64, missing: 0
Pclass: int64, missing: 0
Name: str, missing: 0
Sex: str, missing: 0
Age: float64, missing: 177
SibSp: int64, missing: 0
Parch: int64, missing: 0
Ticket: str, missing: 0
Fare: float64, missing: 0
Cabin: str, missing: 687
Embarked: str, missing: 2
Dropped columns: PassengerId, Name, Ticket, Cabin
Imputed 177 missing Age values with median: 28.0
Imputed 2 missing Embarked values with mode: 'S'
Encoded Sex column as binary: male=0, female=1
Final dataframe shape: (891, 8)


/var/folders/_q/6mjnh6wn7fjglg9lf_wstysw0000gn/T/ipykernel_67274/695905992.py:20: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df['Age'].fillna(median_age, inplace=True)
/var/folders/_q/6mjnh6wn7fjglg9lf_wstysw0000gn/T/ipykernel_67274/695905992.py:26: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series th

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Load the cleaned dataset
df = pd.read_csv('titanic_clean.csv')

# Split into features (X) and target (y)
X = df.drop('Survived', axis=1)
y = df['Survived']

# Perform stratified 80/20 train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Train Logistic Regression classifier on training set
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

# Evaluate on test set
y_pred = model.predict(X_test)

# Print metrics
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision (weighted): {precision_score(y_test, y_pred, average='weighted'):.4f}")
print(f"Recall (weighted): {recall_score(y_test, y_pred, average='weighted'):.4f}")
print(f"F1 Score (weighted): {f1_score(y_test, y_pred, average='weighted'):.4f}")

# Print confusion matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

ValueError: could not convert string to float: 'S'

In [5]:
import matplotlib
matplotlib.use('Agg')
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42

# Load the dataset
df = pd.read_csv('titanic_clean.csv')

# Print shape, column names and dtypes, and describe
print("Shape:", df.shape)
print("\nColumn names and dtypes:")
print(df.dtypes)
print("\nDescribe:")
print(df.describe())

# Compute and print survival rates
print("\nSurvival rates by Sex:")
survival_by_sex = df.groupby('Sex')['Survived'].mean() * 100
print(f"Male: {survival_by_sex[0]:.1f}%")
print(f"Female: {survival_by_sex[1]:.1f}%")

print("\nSurvival rates by Pclass:")
survival_by_pclass = df.groupby('Pclass')['Survived'].mean() * 100
for pclass in sorted(survival_by_pclass.index):
    print(f"Class {pclass}: {survival_by_pclass[pclass]:.1f}%")

# Generate and save plots
# a. Bar chart of Survived counts
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='Survived')
plt.title('Survival Count')
plt.savefig('eda_survival_count.png')
print("Saved eda_survival_count.png")

# b. Histogram of Age with 20 bins, coloured by Survived
plt.figure(figsize=(8, 6))
sns.histplot(data=df, x='Age', bins=20, hue='Survived', multiple='stack')
plt.title('Age Distribution by Survival')
plt.savefig('eda_age_distribution.png')
print("Saved eda_age_distribution.png")

# c. Grouped bar chart of survival rate by Sex
plt.figure(figsize=(6, 4))
sns.barplot(data=df, x='Sex', y='Survived', estimator=lambda x: sum(x)/len(x)*100)
plt.ylabel('Survival Rate (%)')
plt.title('Survival Rate by Sex')
plt.savefig('eda_survival_by_sex.png')
print("Saved eda_survival_by_sex.png")

# d. Correlation heatmap
plt.figure(figsize=(8, 6))
corr = df.corr()
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.savefig('eda_correlation_heatmap.png')
print("Saved eda_correlation_heatmap.png")

Shape: (891, 8)

Column names and dtypes:
Survived      int64
Pclass        int64
Sex           int64
Age         float64
SibSp         int64
Parch         int64
Fare        float64
Embarked        str
dtype: object

Describe:
         Survived      Pclass         Sex         Age       SibSp       Parch  \
count  891.000000  891.000000  891.000000  714.000000  891.000000  891.000000   
mean     0.383838    2.308642    0.352413   29.699118    0.523008    0.381594   
std      0.486592    0.836071    0.477990   14.526497    1.102743    0.806057   
min      0.000000    1.000000    0.000000    0.420000    0.000000    0.000000   
25%      0.000000    2.000000    0.000000   20.125000    0.000000    0.000000   
50%      0.000000    3.000000    0.000000   28.000000    0.000000    0.000000   
75%      1.000000    3.000000    1.000000   38.000000    1.000000    0.000000   
max      1.000000    3.000000    1.000000   80.000000    8.000000    6.000000   

             Fare  
count  891.000000  
mea

ValueError: could not convert string to float: 'S'